# 02J — Functional scores (REVEL/meta_lr)


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 REVEL histogram


In [3]:
tmp = d[['revel_num']].dropna()
fig = px.histogram(tmp, x='revel_num', nbins=50, title='REVEL histogram')
fig.show()


## 2. 📊 meta_lr histogram


In [4]:
tmp = d[['meta_lr_num']].dropna()
fig = px.histogram(tmp, x='meta_lr_num', nbins=50, title='meta_lr histogram')
fig.show()


## 3. 📊 REVEL vs pathogenic


In [5]:
tmp = d[['pathogenic', 'revel_num']].dropna()
fig = px.box(tmp, x='pathogenic', y='revel_num', points='outliers', title='REVEL vs pathogenic')
fig.show()


## 4. 🧪 Mann–Whitney ⭐


In [6]:
g1, g0, stat, p = mann_whitney_bool(d, 'pathogenic', 'revel_num')
display(pd.Series({'n_pathogenic': len(g1), 'n_non_pathogenic': len(g0), 'u_statistic': stat, 'p_value': p}).to_frame('value'))


,value
n_pathogenic,8.500000e+01
n_non_pathogenic,2.188000e+03
u_statistic,1.278455e+05
p_value,2.658067e-09


## 5. 📊 meta_lr vs pathogenic


In [7]:
tmp = d[['pathogenic', 'meta_lr_num']].dropna()
fig = px.box(tmp, x='pathogenic', y='meta_lr_num', points='outliers', title='meta_lr vs pathogenic')
fig.show()


## 6. 🧪 Mann–Whitney


In [8]:
g1, g0, stat, p = mann_whitney_bool(d, 'pathogenic', 'meta_lr_num')
display(pd.Series({'n_pathogenic': len(g1), 'n_non_pathogenic': len(g0), 'u_statistic': stat, 'p_value': p}).to_frame('value'))


,value
n_pathogenic,8.500000e+01
n_non_pathogenic,2.188000e+03
u_statistic,1.233670e+05
p_value,3.106934e-07


## 7. 📊 REVEL vs phenotype


In [9]:
tmp = d[['phenotype', 'revel_num']].dropna()
fig = px.box(tmp, x='phenotype', y='revel_num', points='outliers', title='REVEL vs phenotype')
fig.show()


## 8. 🧪 Kruskal


In [10]:
names, groups, stat, p = kruskal_group(d, 'phenotype', 'revel_num')
display(pd.Series({'groups': ', '.join([str(x) for x in names]), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
groups,"BMD, DMD"
kruskal_h,4.217865
p_value,0.04


## 9. 📊 REVEL vs meta_lr


In [11]:
tmp = d[['revel_num', 'meta_lr_num']].dropna()
fig = px.scatter(tmp, x='revel_num', y='meta_lr_num', opacity=0.5, title='REVEL vs meta_lr')
fig.show()


## 10. 🧪 Spearman ⭐


In [12]:
tmp, rho, p = spearman_xy(d, 'revel_num', 'meta_lr_num')
display(pd.Series({'n': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n,2.273000e+03
spearman_rho,6.105829e-01
p_value,2.416981e-232


## 11. 📊 REVEL bins vs pathogenic


In [13]:
tmp = d[['revel_num', 'pathogenic']].dropna().copy()
tmp['revel_bin'] = pd.cut(tmp['revel_num'], bins=[0, 0.25, 0.5, 0.75, 1.0], include_lowest=True)
tbl = tmp.groupby('revel_bin').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('pathogenic', 'size')).reset_index()
tbl['revel_bin'] = tbl['revel_bin'].astype(str)
display(tbl)
fig = px.bar(tbl, x='revel_bin', y='pathogenic_fraction', hover_data=['n'], title='REVEL bins vs pathogenic fraction')
fig.show()


/var/folders/1g/p5z0kx3523901dvtzvk0_gn00000gn/T/ipykernel_70396/1681462826.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,revel_bin,pathogenic_fraction,n
0,"(-0.001, 0.25]",0.023720,1602
1,"(0.25, 0.5]",0.041026,390
2,"(0.5, 0.75]",0.071942,139
3,"(0.75, 1.0]",0.147887,142


## 12. 📊 meta_lr bins vs pathogenic


In [14]:
tmp = d[['meta_lr_num', 'pathogenic']].dropna().copy()
tmp['meta_lr_bin'] = pd.cut(tmp['meta_lr_num'], bins=[0, 0.25, 0.5, 0.75, 1.0], include_lowest=True)
tbl = tmp.groupby('meta_lr_bin').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('pathogenic', 'size')).reset_index()
tbl['meta_lr_bin'] = tbl['meta_lr_bin'].astype(str)
display(tbl)
fig = px.bar(tbl, x='meta_lr_bin', y='pathogenic_fraction', hover_data=['n'], title='meta_lr bins vs pathogenic fraction')
fig.show()


/var/folders/1g/p5z0kx3523901dvtzvk0_gn00000gn/T/ipykernel_70396/3312252434.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,meta_lr_bin,pathogenic_fraction,n
0,"(-0.001, 0.25]",0.023158,1425
1,"(0.25, 0.5]",0.049505,505
2,"(0.5, 0.75]",0.064516,186
3,"(0.75, 1.0]",0.095541,157


## 13. 📊 REVEL vs allele_freq


In [15]:
tmp = d[['revel_num', 'allele_freq_num']].dropna().copy()
fig = px.scatter(tmp, x='allele_freq_num', y='revel_num', opacity=0.5, title='REVEL vs allele_freq')
fig.update_xaxes(type='log')
fig.show()


## 14. 🧪 Spearman


In [16]:
tmp, rho, p = spearman_xy(d, 'revel_num', 'allele_freq_num')
display(pd.Series({'n': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n,1488.000000
spearman_rho,-0.020155
p_value,0.437231
